# CNNs — LeNet to ResNet Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: LeNet-5

A minimal, faithful LeNet. Tanh activations, average pooling. The only concession to modernity is that we use `nn.CrossEntropyLoss` downstream instead of the original Gaussian connections.

In [ ]:
```python

import torch

import torch.nn as nn

import torch.nn.functional as F

class LeNet5(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)

        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)

        self.pool = nn.AvgPool2d(2)

        self.fc1 = nn.Linear(16 * 5 * 5, 120)

        self.fc2 = nn.Linear(120, 84)

        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x):

        x = self.pool(torch.tanh(self.conv1(x)))

        x = self.pool(torch.tanh(self.conv2(x)))

        x = torch.flatten(x, 1)

        x = torch.tanh(self.fc1(x))

        x = torch.tanh(self.fc2(x))

        return self.fc3(x)

net = LeNet5()

x = torch.randn(1, 1, 32, 32)

print(f"output: {net(x).shape}")

print(f"params: {sum(p.numel() for p in net.parameters()):,}")

In [ ]:
```

Expected output: `output: torch.Size([1, 10])`, `params: 61,706`. That is the entire digit classifier that started modern vision.

### Step 2: A VGG block

One reusable block: two 3x3 convs, ReLU, batch norm, max pool.

In [ ]:
```python

class VGGBlock(nn.Module):

    def __init__(self, in_c, out_c):

        super().__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1)

        self.bn1 = nn.BatchNorm2d(out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)

        self.bn2 = nn.BatchNorm2d(out_c)

        self.pool = nn.MaxPool2d(2)

    def forward(self, x):

        x = F.relu(self.bn1(self.conv1(x)))

        x = F.relu(self.bn2(self.conv2(x)))

        return self.pool(x)

class MiniVGG(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.stack = nn.Sequential(

            VGGBlock(3, 32),

            VGGBlock(32, 64),

            VGGBlock(64, 128),

        )

        self.head = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),

            nn.Linear(128, num_classes),

        )

    def forward(self, x):

        return self.head(self.stack(x))

net = MiniVGG()

x = torch.randn(1, 3, 32, 32)

print(f"output: {net(x).shape}")

print(f"params: {sum(p.numel() for p in net.parameters()):,}")

In [ ]:
```

Three VGG blocks on CIFAR-sized input, an adaptive pool, one linear layer. ~290k parameters. Plenty for CIFAR-10.

### Step 3: A ResNet BasicBlock

The core building block of ResNet-18 and ResNet-34.

In [ ]:
```python

class BasicBlock(nn.Module):

    def __init__(self, in_c, out_c, stride=1):

        super().__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=False)

        self.bn1 = nn.BatchNorm2d(out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, stride=1, padding=1, bias=False)

        self.bn2 = nn.BatchNorm2d(out_c)

        if stride != 1 or in_c != out_c:

            self.shortcut = nn.Sequential(

                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False),

                nn.BatchNorm2d(out_c),

            )

        else:

            self.shortcut = nn.Identity()

    def forward(self, x):

        out = F.relu(self.bn1(self.conv1(x)))

        out = self.bn2(self.conv2(out))

        out = out + self.shortcut(x)

        return F.relu(out)

In [ ]:
```

`bias=False` on conv layers is a batch-norm convention — BN's beta parameter already handles the bias, so carrying conv bias as well is a waste. The `shortcut` only needs a real conv when stride or channel count changes; otherwise it is a no-op identity.

### Step 4: A tiny ResNet

Stack four groups of BasicBlocks to get a working ResNet for CIFAR-sized inputs.

In [ ]:
```python

class TinyResNet(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),

            nn.BatchNorm2d(32),

            nn.ReLU(inplace=True),

        )

        self.layer1 = self._make_group(32, 32, num_blocks=2, stride=1)

        self.layer2 = self._make_group(32, 64, num_blocks=2, stride=2)

        self.layer3 = self._make_group(64, 128, num_blocks=2, stride=2)

        self.layer4 = self._make_group(128, 256, num_blocks=2, stride=2)

        self.head = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),

            nn.Linear(256, num_classes),

        )

    def _make_group(self, in_c, out_c, num_blocks, stride):

        blocks = [BasicBlock(in_c, out_c, stride=stride)]

        for _ in range(num_blocks - 1):

            blocks.append(BasicBlock(out_c, out_c, stride=1))

        return nn.Sequential(*blocks)

    def forward(self, x):

        x = self.stem(x)

        x = self.layer1(x)

        x = self.layer2(x)

        x = self.layer3(x)

        x = self.layer4(x)

        return self.head(x)

net = TinyResNet()

x = torch.randn(1, 3, 32, 32)

print(f"output: {net(x).shape}")

print(f"params: {sum(p.numel() for p in net.parameters()):,}")

In [ ]:
```

Four groups of two blocks each. Stride 2 at the start of groups 2, 3, 4. Channel count doubles at every downsample. Roughly 2.8M parameters. That is the standard recipe that scales cleanly up to ResNet-152.

### Step 5: Compare parameter-to-feature efficiency

Run the same input through all three networks and compare parameter counts.

In [ ]:
```python

def summary(name, net, x):

    y = net(x)

    params = sum(p.numel() for p in net.parameters())

    print(f"{name:12s}  input {tuple(x.shape)} -> output {tuple(y.shape)}  params {params:>10,}")

x = torch.randn(1, 3, 32, 32)

summary("LeNet5",     LeNet5(),       torch.randn(1, 1, 32, 32))

summary("MiniVGG",    MiniVGG(),      x)

summary("TinyResNet", TinyResNet(),   x)

In [ ]:
```

Three models, three eras, three orders of magnitude in parameter count. For CIFAR-10 accuracy, you need roughly: LeNet 60%, MiniVGG 89%, TinyResNet 93% after a few epochs of training.

## Exercises

In [ ]:
1. **(Easy)** Count parameters by hand for `TinyResNet` layer by layer. Compare against `sum(p.numel() for p in net.parameters())`. Where does the majority of the parameter budget go — convs, BN, or the classifier head?
2. **(Medium)** Implement the Bottleneck block (1x1 -> 3x3 -> 1x1 with skip) and use it to build a ResNet-50-style network for CIFAR. Compare params against `TinyResNet`.
3. **(Hard)** Remove the skip connection from `BasicBlock`, train a 34-block "plain" network and a 34-block ResNet on CIFAR-10 for 10 epochs each. Plot training loss vs epoch for both. Reproduce the He et al. Figure 1 result where the plain deep network converges to higher loss than its shallower twin.